# 06 — Dashboard Export & Integration QA
Owners: Rafa + Mutia | World Bank Global Findex 2025, Indonesia 2024

This notebook reads validated aggregate outputs from notebooks 03, 04 and 05; verifies the handoff; and writes Looker Studio-ready CSVs into a private Google Drive folder. In Google Colab choose Runtime → Run all. It does not need the raw ZIP or redistribute respondent-level data.

Pages: 1) Financial inclusion overview, 2) Financial inclusion gaps, 3) Digital adoption, 4) Financial barriers.
The released aggregates only support one segmentation dimension at a time. Never combine gender and income filters on marginal tables.

## 1. Connect the private processed output folder and validate upstream QA
Run Notebook 03, 04 and 05 first. Their output files must all be located in MyDrive/Financial_Inclusion_Private/processed.

In [ ]:
from pathlib import Path
import json, sys
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
CUSTOM_PROCESSED_DIR = None  # Change for local Jupyter execution.
CUSTOM_EXPORT_DIR = None
PRIVATE_PROCESSED = Path("/content/drive/MyDrive/Financial_Inclusion_Private/processed")
PRIVATE_EXPORT = Path("/content/drive/MyDrive/Financial_Inclusion_Private/dashboard_exports")
REQUIRED = [
    "integration_manifest.json","integration_quality_checks.csv",
    "dashboard_kpi_aggregates.csv","dashboard_barriers_aggregates.csv",
    "person1_overview_kpis.csv","person1_account_segments.csv",
    "person1_cross_metrics.csv","person1_gap_summary.csv",
    "person1_notebook03_validation.csv",
    "person2_digital_adoption_aggregates.csv","person2_digital_national_kpis.csv",
    "person2_digital_gap_summary.csv","person2_barrier_aggregates.csv",
    "person2_notebook03_validation.csv","person2_quality_checks.csv",
    "person2_eda_manifest.json"
]
ZIP_SHA = "fc50f0a843e289a3b9784ac81dce1018a8195388608575e6c8315c195073a64c"
CSV_SHA = "77a50e351c16946ad107c08494efe9e8972ba628168fa385b771695971710093"
def sources():
    options = ([Path(CUSTOM_PROCESSED_DIR).expanduser()] if CUSTOM_PROCESSED_DIR else [])
    if IN_COLAB: options.append(PRIVATE_PROCESSED)
    options.extend([Path.cwd()/"data"/"processed", Path.cwd()])
    return options
def full(p):
    return p.is_dir() and all((p/name).is_file() for name in REQUIRED)
SRC = next((p for p in sources() if full(p)), None)
if SRC is None and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SRC = next((p for p in sources() if full(p)), None)
if SRC is None:
    missing = [(str(p), [f for f in REQUIRED if not (p/f).is_file()]) for p in sources() if p.is_dir()]
    raise FileNotFoundError("Run 03, 04 and 05, or set CUSTOM_PROCESSED_DIR. Missing: "+str(missing[:2]))
load_csv = lambda file: pd.read_csv(SRC/file)
m03 = json.loads((SRC/"integration_manifest.json").read_text(encoding="utf-8"))
m05 = json.loads((SRC/"person2_eda_manifest.json").read_text(encoding="utf-8"))
q03 = load_csv("integration_quality_checks.csv")
q04 = load_csv("person1_notebook03_validation.csv")
q05 = load_csv("person2_quality_checks.csv")
q05_detail = load_csv("person2_notebook03_validation.csv")
canon = load_csv("dashboard_kpi_aggregates.csv")
bcanon = load_csv("dashboard_barriers_aggregates.csv")
p1_over = load_csv("person1_overview_kpis.csv")
p1_seg = load_csv("person1_account_segments.csv")
p1_cross = load_csv("person1_cross_metrics.csv")
p1_gap = load_csv("person1_gap_summary.csv")
p2_adopt = load_csv("person2_digital_adoption_aggregates.csv")
p2_nat = load_csv("person2_digital_national_kpis.csv")
p2_gap = load_csv("person2_digital_gap_summary.csv")
p2_bar = load_csv("person2_barrier_aggregates.csv")
assert m03.get("source_zip_sha256") == ZIP_SHA and m03.get("source_csv_sha256") == CSV_SHA
assert m05.get("source_zip_sha256") == ZIP_SHA and m05.get("source_csv_sha256") == CSV_SHA
assert m03.get("qa_passed") is True and m05.get("notebook05_qa_passed") is True
assert m05.get("digital_kpi_cross_checks_passed") == 100 and m05.get("barrier_cross_checks_passed") == 6
assert len(q03)==14 and q03.status.eq("PASS").all()
assert len(q04)==140 and q04.status.eq("PASS").all()
assert len(q05)==11 and q05.status.eq("PASS").all()
assert len(q05_detail)==106 and q05_detail.status.eq("PASS").all()
print("UPSTREAM QA VERIFIED: 14/14 (03), 140/140 (04), 100+6 and 11/11 (05)")
print("Private source:", SRC)

## 2. Verify weighted numerators, denominators, labels, and all analyst handoffs

Rates are percentages from 0 to 100. The survey weight is wgt; n_valid is unweighted respondent count. Groups with n below 30 may not be exported; n from 30 to 49 requires a caution label.

In [ ]:
DIGITAL = ["anydigpayment","internet_use","account_mob","saved","borrowed"]
INCLUSION = ["account","account_fin","account_mob","anydigpayment","saved","borrowed"]
LABEL = {
    "account":"Any account ownership","account_fin":"Financial institution account",
    "account_mob":"Mobile money account","anydigpayment":"Digital payment use",
    "internet_use":"Internet use","saved":"Saved money","borrowed":"Borrowed money"
}
KPI_COLS = ["metric","metric_label","dimension","category","sort_order","numerator_wgt",
            "denominator_wgt","rate_pct","n_valid","n_yes"]
BAR_COLS = ["rank","barrier_code","barrier_label","numerator_wgt",
            "denominator_wgt","rate_pct","n_valid","n_yes"]
def kpi_validate(frame, title):
    assert set(KPI_COLS).issubset(frame.columns), title+" schema mismatch"
    assert frame[KPI_COLS].notna().all().all(), title+" required missing"
    assert frame.n_valid.ge(30).all(), title+" small cell"
    assert frame.denominator_wgt.gt(0).all(), title+" invalid denominator"
    assert frame.rate_pct.between(0,100).all(), title+" out-of-bounds rate"
    assert frame.n_yes.between(0,frame.n_valid).all(), title+" invalid respondent count"
    assert not frame.duplicated(["metric","dimension","category"]).any()
    assert np.allclose(frame.rate_pct,100*frame.numerator_wgt/frame.denominator_wgt,
                       atol=1e-5,rtol=0), title+" weighted rate mismatch"
kpi_validate(canon,"03")
kpi_validate(p2_adopt,"05 digital")
kpi_validate(p2_nat,"05 national")
assert len(canon)==140 and len(p2_adopt)==100 and len(p2_nat)==5
assert len(p1_over)==6 and len(p1_seg)==17 and len(p1_cross)==54 and len(p1_gap)==5
assert len(p2_gap)==4 and len(bcanon)==6 and len(p2_bar)==6
assert set(canon.metric)==set(LABEL)
assert set(p2_adopt.metric)==set(DIGITAL)
def validate_join(left,right,keys,columns,n,title):
    both=left.merge(right,on=keys,how="outer",indicator=True,
                    suffixes=("_c","_a"),validate="one_to_one")
    assert len(both)==n and both["_merge"].eq("both").all(),title+" missing/extra key"
    for col in columns:
        assert np.allclose(both[col+"_c"],both[col+"_a"],atol=1e-5,rtol=0),title+" "+col
    print(title+":",n,"rows matched")
    return both
KEYS=["metric","dimension","category"]
match_digital=validate_join(
    canon[canon.metric.isin(DIGITAL)],p2_adopt,KEYS,
    ["rate_pct","numerator_wgt","denominator_wgt","n_valid","n_yes","sort_order"],
    100,"05 digital vs 03")
match_national=validate_join(
    canon[canon.dimension.eq("national")&canon.metric.isin(DIGITAL)],
    p2_nat,KEYS,["rate_pct","n_valid","n_yes"],5,"05 national vs 03")
match_inclusion=validate_join(
    canon[canon.dimension.eq("national")&canon.metric.isin(INCLUSION)],
    p1_over,["metric"],["rate_pct","n_valid","numerator_wgt","denominator_wgt"],
    6,"04 overview vs 03")
for name,frame in [("04 segment",p1_seg),("04 cross",p1_cross)]:
    match=frame.merge(canon,left_on=["metric","dimension","group"],
                      right_on=["metric","dimension","category"],
                      how="left",suffixes=("_p1","_03"),validate="one_to_one")
    assert match.rate_pct_03.notna().all(),name+" missing match"
    assert np.allclose(match.rate_pct_p1,match.rate_pct_03,atol=1e-5,rtol=0),name+" rates"
match_bar=bcanon.merge(p2_bar,on="barrier_code",how="outer",indicator=True,
                      suffixes=("_03","_05"),validate="one_to_one")
assert len(match_bar)==6 and match_bar["_merge"].eq("both").all()
for col in ["rank","n_valid","n_yes","rate_pct","numerator_wgt","denominator_wgt"]:
    assert np.allclose(match_bar[col+"_03"],match_bar[col+"_05"],atol=1e-5,rtol=0),col
assert bcanon.iloc[0].barrier_code=="fin11d" and p2_bar.barrier_label_id.notna().all()
for gap in p2_gap.itertuples(index=False):
    a=canon.loc[canon.metric.eq(gap.metric)&canon.dimension.eq(gap.dimension)&
                canon.category.eq(gap.group_a)]
    b=canon.loc[canon.metric.eq(gap.metric)&canon.dimension.eq(gap.dimension)&
                canon.category.eq(gap.group_b)]
    assert len(a)==len(b)==1
    assert np.isclose(gap.rate_a_pct,float(a.rate_pct.iloc[0]),atol=1e-5)
    assert np.isclose(gap.rate_b_pct,float(b.rate_pct.iloc[0]),atol=1e-5)
    assert np.isclose(gap.gap_pp,gap.rate_a_pct-gap.rate_b_pct,atol=1e-5)
print("Notebook 04 segments and Notebook 05 barriers/gaps: verified")

## 3. Build four dashboard page tables and two gap summaries

One row is one aggregate, not one respondent. Scores are not additive. Page 1 uses national-only KPI cards; pages 2 and 3 are single-dimension segments; page 4 uses financial non-account holders only (eligible n = 504).

In [ ]:
def decorate(frame,page):
    out=frame[KPI_COLS].copy()
    out.insert(0,"dashboard_page",page)
    out["metric_label"]=out.metric.map(LABEL)
    out["small_sample_warning"]=out.n_valid.between(30,49)
    out["survey_year"]=2024
    out["geography"]="Indonesia"
    out["aggregation_scope"]="one_dimension_only"
    return out
page1=decorate(canon[canon.dimension.eq("national")&canon.metric.isin(INCLUSION)],
               "01_overview")
page2=decorate(canon[canon.metric.eq("account")&canon.dimension.isin(
    ["gender","age_group","education","income_quintile","residence"])],
    "02_inclusion_gap")
page3=decorate(canon[canon.metric.isin(DIGITAL)],"03_digital_adoption")
page4=p2_bar[["rank","barrier_code","barrier_label","barrier_label_id",
              "numerator_wgt","denominator_wgt","rate_pct","n_valid","n_yes"]].copy()
page4.insert(0,"dashboard_page","04_financial_barriers")
page4["survey_year"]=2024
page4["geography"]="Indonesia"
page4["eligible_population"]="account_fin == 0"
page4["eligible_n_unweighted"]=504
page4["denominator_rule"]="eligible, valid Yes(1)/No(2) responses; exclude 3/4"
page4["multiple_response"]=True
page4["small_sample_warning"]=page4.n_valid.between(30,49)
gap2=p1_gap.copy()
gap2.insert(0,"dashboard_page","02_inclusion_gap")
gap2["metric"]="account"
gap2["gap_unit"]="percentage_points"
gap3=p2_gap.copy()
gap3.insert(0,"dashboard_page","03_digital_adoption")
gap3["gap_unit"]="percentage_points"
all_kpis=pd.concat([page1,page2,page3],ignore_index=True)
assert [len(page1),len(page2),len(page3),len(page4),len(gap2),len(gap3)]==[6,17,100,6,5,4]
assert all_kpis.n_valid.ge(30).all() and all_kpis.rate_pct.between(0,100).all()
assert not all_kpis.duplicated(["dashboard_page","metric","dimension","category"]).any()
assert all_kpis.metric_label.notna().all()
display(page1[["metric_label","rate_pct","n_valid"]].round(3))
display(page3.loc[page3.dimension.eq("national"),["metric_label","rate_pct","n_valid"]].round(3))
display(page4[["rank","barrier_label_id","rate_pct","n_valid"]].round(3))
print("Page exports prepared:",[len(page1),len(page2),len(page3),len(page4)])

## 4. Final export QA and dashboard data dictionary

Do not export microdata. Only aggregate tables are written. For a Looker Studio scorecard, select one metric and category instead of summing or averaging percentage rows. Preserve the original weighted numerator and denominator for auditability.

In [ ]:
checks=[
 ("03 QA",len(q03),14,len(q03)==14 and q03.status.eq("PASS").all()),
 ("04 QA",len(q04),140,len(q04)==140 and q04.status.eq("PASS").all()),
 ("05 detailed QA",len(q05_detail),106,len(q05_detail)==106 and q05_detail.status.eq("PASS").all()),
 ("05 overall QA",len(q05),11,len(q05)==11 and q05.status.eq("PASS").all()),
 ("Page 1 rows",len(page1),6,len(page1)==6),
 ("Page 2 rows",len(page2),17,len(page2)==17),
 ("Page 3 rows",len(page3),100,len(page3)==100),
 ("Page 4 rows",len(page4),6,len(page4)==6),
 ("Inclusion gaps",len(gap2),5,len(gap2)==5),
 ("Digital gaps",len(gap3),4,len(gap3)==4),
 ("KPI cells below 30",int(all_kpis.n_valid.lt(30).sum()),0,all_kpis.n_valid.ge(30).all()),
 ("Duplicate page keys",int(all_kpis.duplicated(
    ["dashboard_page","metric","dimension","category"]).sum()),0,
    not all_kpis.duplicated(["dashboard_page","metric","dimension","category"]).any()),
]
qa06=pd.DataFrame([
    {"check":name,"actual":actual,"expected":expected,
     "status":"PASS" if passed else "FAIL"}
    for name,actual,expected,passed in checks])
display(qa06)
if not qa06.status.eq("PASS").all():
    raise AssertionError("Notebook 06 QA failed; STOP before dashboard export")
print(f"ALL {len(qa06)} DASHBOARD EXPORT CHECKS PASSED.")

In [ ]:
if CUSTOM_EXPORT_DIR:
    DEST=Path(CUSTOM_EXPORT_DIR).expanduser()
elif IN_COLAB and SRC==PRIVATE_PROCESSED:
    DEST=PRIVATE_EXPORT
else:
    DEST=SRC.parent/"dashboard_exports"
assert DEST.resolve()!=SRC.resolve(),"Export must be separate from private source data"
DEST.mkdir(parents=True,exist_ok=True)
dictionary=pd.DataFrame([
    ("dashboard_page","One of four dashboard pages; NOT a population dimension"),
    ("metric","Original binary indicator name"),
    ("dimension","Only one segmentation dimension per aggregate"),
    ("category","Single group label inside dimension"),
    ("rate_pct","Weighted rate in 0–100 units, not addable across groups"),
    ("numerator_wgt","Sum of weights for Yes(1) among valid respondents"),
    ("denominator_wgt","Sum of weights for eligible valid responses"),
    ("n_valid","Unweighted valid sample count; min dashboard n=30"),
    ("n_yes","Unweighted count of respondents answering Yes"),
    ("small_sample_warning","True for unweighted n=30–49"),
    ("gap_pp","Rate of comparison A minus B in percentage points"),
    ("barrier_code","fin11a–f, only non-account holders account_fin=0"),
    ("eligible_n_unweighted","504 account_fin=0 respondents; valid n differs per item"),
],columns=["field","definition"])
tables={
 "dashboard_page1_overview.csv":page1,
 "dashboard_page2_inclusion_segments.csv":page2,
 "dashboard_page3_digital_adoption.csv":page3,
 "dashboard_page4_financial_barriers.csv":page4,
 "dashboard_page2_gap_summary.csv":gap2,
 "dashboard_page3_gap_summary.csv":gap3,
 "dashboard_all_kpis.csv":all_kpis,
 "dashboard_export_quality_checks.csv":qa06,
 "dashboard_data_dictionary.csv":dictionary,
}
for filename,frame in tables.items():
    frame.to_csv(DEST/filename,index=False,float_format="%.8f")
manifest={
 "project":"Indonesia-Financial-Inclusion-Digital-Finance-Analytics",
 "notebook":"06_dashboard_export.ipynb",
 "country":"Indonesia","survey_year":2024,
 "source_zip_sha256":ZIP_SHA,"source_csv_sha256":CSV_SHA,
 "upstream_03_qa_passed":True,"upstream_04_qa_passed":True,"upstream_05_qa_passed":True,
 "notebook06_qa_passed":True,"notebook06_check_count":len(qa06),
 "min_n_dashboard":30,"respondent_rows_exported":False,
 "supports_simultaneous_demographic_filters":False,
 "generated_at_utc":datetime.now(timezone.utc).isoformat(),
 "tables":{f:int(len(df)) for f,df in tables.items()}
}
(DEST/"dashboard_export_manifest.json").write_text(
    json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
print("NOTEBOOK 06 COMPLETED: ALL DASHBOARD EXPORT CHECKS PASSED.")
print("Saved to private directory:",DEST)
for f,frame in tables.items(): print(" -",f,":",len(frame),"rows")
print(" - dashboard_export_manifest.json")
print("Do not publish respondent-level files or the source ZIP.")

## 5. Google Sheets and Looker Studio handoff

1. In Drive open Financial_Inclusion_Private/dashboard_exports and check dashboard_export_quality_checks.csv.
2. Create one Google Sheets spreadsheet with separate tabs from the six dashboard page/gap CSVs. Only import aggregate-only CSVs.
3. Connect Google Sheets tabs to Looker Studio. Page 1: national scorecards; Page 2: account ownership by one demographic dimension and gaps; Page 3: digital adoption and payment gaps; Page 4: barriers.
4. Configure chart metrics from rate_pct, which already ranges from 0–100. Use number display with a percent suffix, or divide by 100 if using Looker Studio percent formatting.
5. Never use SUM(rate_pct) or AVG(rate_pct) across different groups. For segment charts filter to one metric and one dimension. Do not use simultaneous gender+income cross-filters with marginal aggregated tables.
6. Share only aggregate workbook/dashboard outputs; include data dictionary, survey limitations, cross-sectional year, descriptive nature of gaps, and education tertiary n=41 caveat.

Expected success marker: NOTEBOOK 06 COMPLETED: ALL DASHBOARD EXPORT CHECKS PASSED.

## 6. Interpretasi hasil akhir Notebook 06 — diverifikasi dari eksekusi Colab, 19 November 2026

**Status pipeline:** Notebook 06 telah dijalankan sampai akhir di Google Colab menggunakan output privat Notebook 03, 04, dan 05. Pemeriksaan **14/14 QA Notebook 03, 140/140 QA Notebook 04, 106/106 QA detail Notebook 05, 11/11 QA Notebook 05, dan 12/12 QA ekspor Notebook 06 semuanya PASS**. Total terdapat **6 baris untuk Page 1; 17 untuk Page 2; 100 untuk Page 3; 6 untuk Page 4**, serta **5 baris gap inklusi** dan **4 baris gap digital**. Tabel gabungan berisi 123 baris KPI, sebagian indikator sengaja muncul lagi di halaman berbeda sehingga **jangan di-SUM/AVG**.

### 6.1 Executive summary (tingkat nasional)

Hasil persentase berbobot responden survei Indonesia 2024:

| Indikator | Rate berbobot |
|---|---:|
| Memiliki rekening jenis apa pun (\`account\`) | **56,328%** |
| Rekening lembaga keuangan (\`account_fin\`) | **52,392%** |
| Rekening mobile money (\`account_mob\`) | **22,013%** |
| Menggunakan pembayaran digital (\`anydigpayment\`) | **42,693%** |
| Menabung (\`saved\`) | **52,995%** |
| Meminjam (\`borrowed\`) | **47,636%** |
| Penggunaan internet (\`internet_use\`, Page 3) | **70,463%** |

**Interpretasi:** Kepemilikan rekening sekitar 56,33% dan penggunaan pembayaran digital sekitar 42,69%, tetapi keduanya memiliki konsep survei yang berbeda dan responden yang dapat tumpang tindih. **Tidak boleh** menyatakan selisihnya sebagai konversi dari pemilik rekening menjadi pengguna digital. Seluruh angka adalah estimasi deskriptif berbobot, **bukan jumlah orang Indonesia**.

### 6.2 Page 1 — Financial Inclusion Overview

Enam scorecard nasional menyajikan gambaran akses dan pemanfaatan layanan keuangan. Rekening lembaga keuangan (**52,39%**) lebih umum daripada rekening mobile money (**22,01%**); menabung (**52,99%**) dan meminjam (**47,64%**) adalah indikator aktivitas berbeda yang dapat tumpang tindih dengan kepemilikan rekening. **Tidak diperbolehkan** menjumlahkan persentase atau menyebutnya funnel.

### 6.3 Page 2 — Inclusion Gap

Analisis Notebook 04 yang terkonfirmasi dalam QA Notebook 06 menemukan perbedaan deskriptif kepemilikan rekening:

| Dimensi | Perbandingan | Gap |
|---|---|---:|
| Pendidikan | Tertiary or more (85,607%) − Primary or less (37,638%) | **47,969 pp** |
| Pendapatan, *maximum–minimum* | Q5 (78,002%) − Q3 (46,203%) | **31,799 pp** |
| Pendapatan, *Q5–Q1 khusus* | Q5 (78,002%) − Q1 (46,425%) | **31,577 pp** |
| Usia | 25–34 (70,135%) − 55+ (39,519%) | **30,616 pp** |
| Gender | Female (58,031%) − Male (54,639%) | **3,392 pp** |
| Domisili | Urban (56,666%) − Rural (56,030%) | **0,636 pp** |

**Interpretasi:** Pendidikan, pendapatan dan usia memperlihatkan ketimpangan observasional lebih besar daripada gender dan urban–rural dalam indikator kepemilikan rekening. **Kelompok pendidikan tertiary hanya n=41** sehingga estimasinya lebih rentan variasi sampel. Pola kuintil tidak monoton karena Q3 lebih rendah daripada Q1; tidak tepat menyatakan setiap kuintil lebih kaya pasti memiliki angka lebih besar. Tabel \`dashboard_page2_gap_summary.csv\` menyimpan *maximum–minimum* per dimensi; Q5–Q1 perlu ditampilkan sebagai pembandingan tambahan terpisah agar maknanya tidak tertukar.

### 6.4 Page 3 — Digital Finance Adoption

| Perbandingan pembayaran digital | Gap |
|---|---:|
| Internet user vs non-internet user | **36,164 pp** |
| Pendapatan Q5 vs Q1 | **33,002 pp** |
| Urban vs rural | **4,666 pp** |
| Female vs male | **1,569 pp** |

Penggunaan internet (**70,46%**) secara deskriptif lebih luas dibanding penggunaan pembayaran digital (**42,69%**). Pada pembandingan *yang terpilih*, status internet dan pendapatan memiliki gap pembayaran digital paling besar. Namun ini **bukan bukti bahwa internet atau pendapatan menyebabkan** adopsi. Jangan menggunakan kontrol filter gender dan pendapatan secara simultan karena sumber dashboard berisi agregat per-dimensi, bukan agregat silang.

### 6.5 Page 4 — Financial Barriers

**Populasi eligible: \`account_fin == 0\`, n=504.** Enam pertanyaan hambatan merupakan *multiple response* dan masing-masing mempunyai denominator jawaban valid berbeda:

| Hambatan | Rate berbobot | n jawaban valid |
|---|---:|---:|
| Dana tidak cukup (\`fin11d\`) | **63,758%** | 502 |
| Anggota keluarga sudah memiliki rekening (\`fin11e\`) | **52,280%** | 493 |
| Biaya layanan mahal (\`fin11b\`) | **51,543%** | 484 |
| Lembaga keuangan terlalu jauh (\`fin11a\`) | **39,737%** | 495 |
| Dokumen yang dibutuhkan kurang (\`fin11c\`) | **33,382%** | 500 |
| Kurang percaya lembaga keuangan (\`fin11f\`) | **17,778%** | 491 |

**Interpretasi:** Keterbatasan dana adalah alasan yang paling banyak dilaporkan di antara enam pertanyaan tersebut. Denominator tiap hambatan hanya menghitung jawaban valid **1=Ya / 2=Tidak** dari responden eligible. Kode 3/4 dikeluarkan. Baris tidak saling eksklusif dan **persentasenya tidak berjumlah 100%**.

### 6.6 Implikasi dan batasan

- **Hipotesis untuk tindak lanjut:** evaluasi akses biaya, literasi/pengalaman digital, dan desain onboarding yang lebih mudah untuk kelompok dengan adopsi rendah. Analisis ini tidak menunjukkan efek intervensi.
- **Metodologi:** survei satu tahun (Indonesia 2024), estimasi berbobot \`wgt\`, bukan time series, peta provinsi, uji signifikansi, atau kausalitas. \`n_valid\` adalah jumlah sampel tidak berbobot; kelompok 30–49 ditandai *small_sample_warning*.
- **Dashboard:** Page 1 dan 2 oleh Rafa, Page 3 dan 4 oleh Mutia; keduanya menguji data, narasi, filter, dan akses bersama. Data di Sheets/Looker harus **agregat saja**.

### 6.7 Catatan penting untuk Google Sheets dan Looker Studio

Pada audit spreadsheet \`Findex Indonesia 2024 — Dashboard Data\` (tab lama \`KPI_Aggregates\`, locale \`in_ID\`) ditemukan nilai \`rate_pct\` hasil impor CSV sebelumnya **rusak akibat tanda titik desimal diperlakukan sebagai pemisah ribuan**. Contoh: rate sesungguhnya **61,99266284** justru tersimpan sebagai **6199266284**. **Jangan** menggunakan tab lama sebagai sumber dashboard. Mengubah tampilan atau format sel saja tidak cukup untuk memperbaiki nilai yang sudah terlanjur berubah; **reimpor dari CSV tervalidasi**.

Langkah aman: simpan tab lama sebagai arsip, ubah **File → Settings → Locale = United States** sebelum mengimpor CSV baru dari folder \`dashboard_exports/\`, gunakan 6 file tabel halaman/gap, lalu verifikasi nilai numerik \`rate_pct\` berada dalam 0–100 (misalnya nasional \`account = 56.328\`, \`anydigpayment = 42.693\`). Jangan mengunggah \`integrated_respondents_private.csv\` atau ZIP mentah. Untuk Looker Studio, pilih satu \`metric\` dan satu \`dimension\` setiap visual; jangan menjumlahkan persentase atau mengaktifkan filter lintas-dimensi pada tabel marginal.

**Output diverifikasi pada run Colab:** \`NOTEBOOK 06 COMPLETED: ALL DASHBOARD EXPORT CHECKS PASSED.\` File agregat dan manifest berada di \`MyDrive/Financial_Inclusion_Private/dashboard_exports/\`. Notebook GitHub menambahkan interpretasi dokumenter; menjalankan ulang hanya diperlukan bila data atau pipeline berubah.